# Attestor 4.3 — SVD Adapter Compression

Decompose the QLoRA adapter weights via **Singular Value Decomposition** to:
1. **Analyze** what the fine-tune learned (singular value spectrum)
2. **Compress** the adapter by truncating low-energy singular values
3. **Extract** dominant security-detection directions as interpretable vectors

Input: LoRA adapter from HuggingFace Hub (`attestor-qwen38-lora`)
Output: Compressed adapter + analysis of learned features

**Runtime: GPU recommended but not required (SVD runs on CPU)**

In [ ]:
# Cell 1: Install dependencies
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'safetensors', 'huggingface_hub', 'matplotlib', 'numpy', 'torch'])
print('Dependencies installed.')

In [ ]:
# Cell 2: Download adapter from HuggingFace Hub
from google.colab import userdata
from huggingface_hub import hf_hub_download, HfApi
import os, json

token = userdata.get('HF_TOKEN')
api = HfApi(token=token)
username = api.whoami()['name']
REPO_ID = f'{username}/attestor-qwen38-lora'

ADAPTER_DIR = '/content/adapter'
os.makedirs(ADAPTER_DIR, exist_ok=True)

# Download all adapter files
files = api.list_repo_files(REPO_ID)
for f in files:
    hf_hub_download(REPO_ID, f, local_dir=ADAPTER_DIR, token=token)
    print(f'  Downloaded: {f}')

print(f'\nAdapter files in {ADAPTER_DIR}:')
for f in os.listdir(ADAPTER_DIR):
    size = os.path.getsize(os.path.join(ADAPTER_DIR, f)) / 1024**2
    print(f'  {f}: {size:.1f} MB')

In [ ]:
# Cell 3: Load adapter weights and inspect structure
from safetensors.torch import load_file, save_file
import torch
import numpy as np

weights = load_file(os.path.join(ADAPTER_DIR, 'adapter_model.safetensors'))

print(f'Adapter has {len(weights)} weight tensors\n')

# Group by layer and module
lora_a = {}  # down-projection matrices
lora_b = {}  # up-projection matrices

for name, tensor in sorted(weights.items()):
    shape = tuple(tensor.shape)
    print(f'  {name}: {shape}')
    if 'lora_A' in name:
        key = name.replace('.lora_A.weight', '').replace('.lora_A.default.weight', '')
        lora_a[key] = tensor.float()
    elif 'lora_B' in name:
        key = name.replace('.lora_B.weight', '').replace('.lora_B.default.weight', '')
        lora_b[key] = tensor.float()

print(f'\nFound {len(lora_a)} LoRA A matrices and {len(lora_b)} LoRA B matrices')
print(f'LoRA rank (r): {list(lora_a.values())[0].shape[0]}')

In [ ]:
# Cell 4: SVD decomposition of each LoRA layer
# LoRA computes: delta_W = B @ A (where A is r×d_in, B is d_out×r)
# SVD of delta_W = U @ diag(S) @ V^T reveals the learned directions

svd_results = {}

for key in sorted(lora_a.keys()):
    if key not in lora_b:
        continue
    A = lora_a[key]  # (r, d_in)
    B = lora_b[key]  # (d_out, r)
    
    # Compute the effective weight update
    delta_W = B @ A  # (d_out, d_in)
    
    # SVD
    U, S, Vh = torch.linalg.svd(delta_W, full_matrices=False)
    
    # Energy distribution
    energy = (S ** 2) / (S ** 2).sum()
    cumulative = torch.cumsum(energy, dim=0)
    
    # Find rank needed for 90%, 95%, 99% energy
    r90 = (cumulative < 0.90).sum().item() + 1
    r95 = (cumulative < 0.95).sum().item() + 1
    r99 = (cumulative < 0.99).sum().item() + 1
    
    layer_name = key.split('.')[-1] if '.' in key else key
    svd_results[key] = {
        'U': U, 'S': S, 'Vh': Vh,
        'energy': energy,
        'cumulative': cumulative,
        'r90': r90, 'r95': r95, 'r99': r99,
        'shape': tuple(delta_W.shape),
        'frobenius': delta_W.norm().item(),
    }
    print(f'{key}:')
    print(f'  delta_W shape: {delta_W.shape}, Frobenius norm: {delta_W.norm():.4f}')
    print(f'  Top-5 singular values: {S[:5].tolist()}')
    print(f'  Rank for 90% energy: {r90}, 95%: {r95}, 99%: {r99} (of {len(S)})')
    print()

In [ ]:
# Cell 5: Visualize singular value spectra
import matplotlib.pyplot as plt

# Group by module type
module_types = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                'gate_proj', 'up_proj', 'down_proj']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, mod_type in enumerate(module_types):
    ax = axes[idx]
    for key, res in svd_results.items():
        if mod_type in key:
            S = res['S'].cpu().numpy()
            ax.semilogy(S / S[0], alpha=0.3, linewidth=0.5)
    ax.set_title(mod_type, fontsize=14, fontweight='bold')
    ax.set_xlabel('Singular value index')
    ax.set_ylabel('Normalized magnitude')
    ax.axhline(y=0.1, color='r', linestyle='--', alpha=0.5, label='10% threshold')
    ax.set_ylim(1e-4, 1.5)

# Summary in last subplot
ax = axes[7]
avg_r90 = np.mean([r['r90'] for r in svd_results.values()])
avg_r95 = np.mean([r['r95'] for r in svd_results.values()])
avg_r99 = np.mean([r['r99'] for r in svd_results.values()])
original_r = list(lora_a.values())[0].shape[0]
ax.bar(['90%', '95%', '99%', 'Original'],
       [avg_r90, avg_r95, avg_r99, original_r],
       color=['#2ecc71', '#f39c12', '#e74c3c', '#3498db'])
ax.set_title('Avg Rank Needed', fontsize=14, fontweight='bold')
ax.set_ylabel('Rank')
for i, v in enumerate([avg_r90, avg_r95, avg_r99, original_r]):
    ax.text(i, v + 0.3, f'{v:.0f}', ha='center', fontweight='bold')

fig.suptitle('SVD Spectrum of Attestor LoRA Adapter — Singular Value Decay per Module',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/svd_spectrum.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nAverage rank needed: 90%={avg_r90:.0f}, 95%={avg_r95:.0f}, 99%={avg_r99:.0f} (original r={original_r})')
print(f'Potential compression: {(1 - avg_r95/original_r)*100:.0f}% at 95% energy retention')

In [ ]:
# Cell 6: Compress adapter via SVD truncation
# Keep only the top-k singular values per layer (95% energy)

ENERGY_THRESHOLD = 0.95  # Change to 0.90 for more compression, 0.99 for less

compressed_a = {}
compressed_b = {}
original_params = 0
compressed_params = 0

for key in sorted(lora_a.keys()):
    if key not in svd_results:
        continue
    
    res = svd_results[key]
    U, S, Vh = res['U'], res['S'], res['Vh']
    cumulative = res['cumulative']
    
    # Find rank for desired energy retention
    k = (cumulative < ENERGY_THRESHOLD).sum().item() + 1
    k = max(k, 1)  # at least rank 1
    
    # Truncated SVD: delta_W ≈ U[:,:k] @ diag(S[:k]) @ Vh[:k,:]
    # Refactor into new LoRA: A_new = sqrt(S[:k]) * Vh[:k,:], B_new = U[:,:k] * sqrt(S[:k])
    sqrt_s = torch.sqrt(S[:k])
    new_A = sqrt_s.unsqueeze(1) * Vh[:k, :]       # (k, d_in)
    new_B = U[:, :k] * sqrt_s.unsqueeze(0)         # (d_out, k)
    
    orig_r = lora_a[key].shape[0]
    orig_count = lora_a[key].numel() + lora_b[key].numel()
    new_count = new_A.numel() + new_B.numel()
    
    original_params += orig_count
    compressed_params += new_count
    
    # Store with original naming convention
    a_key = key + '.lora_A.weight'
    b_key = key + '.lora_B.weight'
    if a_key not in weights:
        a_key = key + '.lora_A.default.weight'
        b_key = key + '.lora_B.default.weight'
    
    compressed_a[a_key] = new_A.to(torch.float16)
    compressed_b[b_key] = new_B.to(torch.float16)
    
    print(f'{key}: r={orig_r} -> r={k} ({(1-new_count/orig_count)*100:.0f}% fewer params)')

ratio = compressed_params / original_params
print(f'\nTotal: {original_params:,} -> {compressed_params:,} params ({(1-ratio)*100:.1f}% reduction)')
print(f'Energy retained: {ENERGY_THRESHOLD*100:.0f}%')

In [ ]:
# Cell 7: Save compressed adapter
COMPRESSED_DIR = '/content/adapter-compressed'
os.makedirs(COMPRESSED_DIR, exist_ok=True)

# Merge compressed weights
compressed_weights = {}
for name, tensor in weights.items():
    if name in compressed_a:
        compressed_weights[name] = compressed_a[name]
    elif name in compressed_b:
        compressed_weights[name] = compressed_b[name]
    else:
        compressed_weights[name] = tensor

save_file(compressed_weights, os.path.join(COMPRESSED_DIR, 'adapter_model.safetensors'))

# Copy config files (update LoRA rank in config)
import shutil
for f in os.listdir(ADAPTER_DIR):
    if f != 'adapter_model.safetensors':
        shutil.copy2(os.path.join(ADAPTER_DIR, f), COMPRESSED_DIR)

# Size comparison
orig_size = os.path.getsize(os.path.join(ADAPTER_DIR, 'adapter_model.safetensors'))
comp_size = os.path.getsize(os.path.join(COMPRESSED_DIR, 'adapter_model.safetensors'))
print(f'Original adapter: {orig_size/1024**2:.1f} MB')
print(f'Compressed adapter: {comp_size/1024**2:.1f} MB')
print(f'Reduction: {(1-comp_size/orig_size)*100:.1f}%')

In [ ]:
# Cell 8: Verify compressed adapter produces similar outputs
# Reconstruction error per layer

print('Reconstruction error (Frobenius norm of difference):\n')
max_error = 0
for key in sorted(svd_results.keys()):
    if key not in lora_a or key not in lora_b:
        continue
    
    # Original delta_W
    orig_dW = lora_b[key] @ lora_a[key]
    
    # Compressed delta_W
    a_key = key + '.lora_A.weight'
    b_key = key + '.lora_B.weight'
    if a_key not in compressed_a:
        a_key = key + '.lora_A.default.weight'
        b_key = key + '.lora_B.default.weight'
    
    comp_dW = compressed_b[b_key].float() @ compressed_a[a_key].float()
    
    error = (orig_dW - comp_dW).norm() / orig_dW.norm()
    max_error = max(max_error, error.item())
    print(f'  {key}: relative error = {error:.6f}')

print(f'\nMax relative error: {max_error:.6f}')
if max_error < 0.1:
    print('Compression is faithful — error is small.')
else:
    print('WARNING: High reconstruction error. Consider higher energy threshold.')

In [ ]:
# Cell 9: Upload compressed adapter to HuggingFace Hub
compressed_repo = f'{username}/attestor-qwen38-lora-svd'
api.create_repo(compressed_repo, exist_ok=True, private=True)
api.upload_folder(
    folder_path=COMPRESSED_DIR,
    repo_id=compressed_repo,
    repo_type='model',
)
print(f'Compressed adapter uploaded to https://huggingface.co/{compressed_repo}')

In [ ]:
# Cell 10: Extract top security directions (interpretable features)
# The top singular vectors represent the strongest learned directions

print('Top-3 learned directions per attention module (by Frobenius contribution):\n')
print('These are the weight-space directions the fine-tune changed most.')
print('High-norm directions in q/k/v projections = learned attention patterns.')
print('High-norm directions in gate/up/down = learned FFN features.\n')

# Rank layers by how much the fine-tune changed them
layer_changes = []
for key, res in svd_results.items():
    layer_changes.append((key, res['frobenius'], res['S'][0].item(), res['r95']))

layer_changes.sort(key=lambda x: -x[1])

print(f'{"Layer":<60s} {"||dW||":>10s} {"sigma_1":>10s} {"r_95%":>8s}')
print('-' * 90)
for key, frob, s1, r95 in layer_changes[:20]:
    print(f'{key:<60s} {frob:>10.4f} {s1:>10.4f} {r95:>8d}')

print(f'\n--- Most changed layers are where the model learned security patterns ---')
print(f'--- These are candidates for rule extraction via probing ---')